# EDA — PETR4.SA (Petrobras)

Análise exploratória dos dados históricos de preço da Petrobras (PETR4.SA).
Objetivo: entender padrões, distribuições e correlações que guiem a modelagem LSTM.

**Disclaimer:** Esta análise NÃO constitui recomendação de investimento.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

%matplotlib inline

## 1. Carregamento dos Dados

In [ ]:
df = pd.read_parquet("../data/processed/petr4_features.parquet")
print(f"Shape: {df.shape}")
print(f"Período: {df.index.min()} a {df.index.max()}")
df.info()

In [ ]:
df.describe().round(2)

## 2. Análise Temporal

Evolução do preço de fechamento e volume ao longo do tempo.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df.index, df["Close"], color="#1f77b4", linewidth=0.8)
axes[0].set_title("PETR4.SA — Preço de Fechamento")
axes[0].set_ylabel("Preço (BRL)")

axes[1].bar(df.index, df["Volume"], color="#2ca02c", alpha=0.6, width=1)
axes[1].set_title("PETR4.SA — Volume")
axes[1].set_ylabel("Volume")
axes[1].set_xlabel("Data")

plt.tight_layout()
plt.show()

In [ ]:
# Preço com Bollinger Bands e Médias Móveis
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df["Close"], label="Close", linewidth=0.8)
ax.plot(df.index, df["sma_20"], label="SMA 20", linewidth=0.7, alpha=0.8)
ax.plot(df.index, df["sma_50"], label="SMA 50", linewidth=0.7, alpha=0.8)
ax.fill_between(
    df.index, df["bollinger_lower"], df["bollinger_upper"],
    alpha=0.15, color="gray", label="Bollinger Bands"
)
ax.set_title("PETR4.SA — Preço com Indicadores")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Distribuições

Análise dos retornos diários — esperamos caudas pesadas (leptocúrtica).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de retornos
axes[0].hist(df["daily_return"], bins=80, density=True, alpha=0.7, color="#1f77b4")
x = np.linspace(df["daily_return"].min(), df["daily_return"].max(), 200)
axes[0].plot(x, stats.norm.pdf(x, df["daily_return"].mean(), df["daily_return"].std()),
             "r-", linewidth=1.5, label="Normal")
axes[0].set_title("Distribuição dos Retornos Diários")
axes[0].legend()

# QQ-Plot
stats.probplot(df["daily_return"].dropna(), dist="norm", plot=axes[1])
axes[1].set_title("QQ-Plot dos Retornos Diários")

plt.tight_layout()
plt.show()

print(f"Skewness: {df['daily_return'].skew():.4f}")
print(f"Kurtosis: {df['daily_return'].kurtosis():.4f}")
print("(Kurtosis > 0 indica caudas pesadas — típico de ativos financeiros)")

## 4. Correlações

Heatmap de correlação entre features técnicas — identificar redundâncias.

In [ ]:
feature_cols = [
    "Close", "Volume", "sma_20", "sma_50", "ema_12", "ema_26",
    "rsi_14", "macd", "macd_signal", "bollinger_upper", "bollinger_lower",
    "volume_sma_20", "daily_return"
]

corr = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, square=True, ax=ax, linewidths=0.5)
ax.set_title("Correlação entre Features Técnicas")
plt.tight_layout()
plt.show()

print("\nFeatures altamente correlacionadas (|r| > 0.95):")
high_corr = corr.abs().unstack()
high_corr = high_corr[high_corr < 1.0]
print(high_corr[high_corr > 0.95].drop_duplicates().sort_values(ascending=False))

## 5. Estacionariedade

Teste ADF (Augmented Dickey-Fuller). Séries de preço geralmente são não-estacionárias;
retornos tendem a ser estacionários.

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_test(series: pd.Series, name: str) -> dict:
    result = adfuller(series.dropna(), autolag="AIC")
    output = {
        "Série": name,
        "ADF Statistic": result[0],
        "p-value": result[1],
        "Estacionária (p<0.05)": result[1] < 0.05,
    }
    return output

adf_results = [
    adf_test(df["Close"], "Close (Preço)"),
    adf_test(df["daily_return"], "daily_return (Retorno)"),
    adf_test(df["rsi_14"], "RSI 14"),
    adf_test(df["macd"], "MACD"),
    adf_test(df["Volume"], "Volume"),
]

pd.DataFrame(adf_results).set_index("Série")

## 6. Sazonalidade

Decomposição sazonal do preço de fechamento (período ~252 trading days/ano).

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(df["Close"], model="multiplicative", period=252)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomposition.observed.plot(ax=axes[0], title="Observado")
decomposition.trend.plot(ax=axes[1], title="Tendência")
decomposition.seasonal.plot(ax=axes[2], title="Sazonalidade")
decomposition.resid.plot(ax=axes[3], title="Resíduo")
plt.tight_layout()
plt.show()

## 7. Outliers

Detecção de outliers em volume e retornos via box plots e IQR.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot(df["daily_return"].dropna(), vert=True)
axes[0].set_title("Box Plot — Retornos Diários")
axes[0].set_ylabel("Retorno")

axes[1].boxplot(df["Volume"], vert=True)
axes[1].set_title("Box Plot — Volume")
axes[1].set_ylabel("Volume")

plt.tight_layout()
plt.show()

# Quantificar outliers (IQR)
for col in ["daily_return", "Volume"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)")

## 8. Insights para o Modelo

### Conclusões da EDA

1. **Não-estacionariedade do preço**: Confirmada pelo teste ADF. O LSTM recebe sequências de preços escalados (MinMaxScaler), o que mitiga parcialmente. Features como RSI e MACD, que são estacionárias, enriquecem o sinal.

2. **Caudas pesadas nos retornos**: Kurtosis elevada indica eventos extremos. O modelo deve ser avaliado com métricas robustas (MAE além de RMSE) e disclaimers de risco são obrigatórios.

3. **Alta correlação entre médias móveis e preço**: SMA/EMA são altamente correlacionadas com Close. Ainda assim, capturam tendências de curto vs. longo prazo que o LSTM pode explorar via atenção temporal.

4. **Volume como sinal complementar**: Volume tem correlação baixa com preço mas picos de volume frequentemente precedem movimentos significativos — feature relevante.

5. **Sequence length = 60**: Com ~252 trading days/ano, 60 dias (~3 meses) captura ciclos de médio prazo sem introduzir ruído excessivo.

6. **Outliers em volume**: Não serão removidos — representam eventos de mercado reais (ex: resultados trimestrais, eventos políticos) que o modelo deve aprender.

7. **Split temporal obrigatório**: Dados de séries temporais não podem ser shuffled. Train/Val/Test em ordem cronológica para evitar data leakage.